In [1]:
#INFERENCE CODE

import torch
from torch.nn import functional as F
from torch.nn import ModuleDict
from dataclasses import dataclass
from transformers import GPT2LMHeadModel
import torch.nn as nn
import inspect
import math
import time
import numpy as np
import os

class CausalAttention(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_attn=nn.Linear(config.n_embd,3*config.n_embd)
                self.c_proj=nn.Linear(config.n_embd,config.n_embd)
                self.n_embd=config.n_embd
                self.n_head=config.n_head
                self.c_proj.NANOGPT_SCALE_INIT=1
                self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))
        def forward(self,x):
                B,T,C=x.shape
                qkv=self.c_attn(x)
                q,k,v=qkv.split(self.n_embd,dim=2)
                q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                
                # att=(q@k.transpose(-2,-1))*(1/math.sqrt(k.size(-1)))
                # att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
                # att=F.softmax(att,dim=-1)
                # y=att@v
                y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
                
                y=y.transpose(1,2).contiguous().view(B,T,C)
                y=self.c_proj(y)
                return y

class MLP(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_fc=nn.Linear(config.n_embd,4*config.n_embd)
                self.gelu=nn.GELU(approximate="tanh")
                self.c_proj=nn.Linear(4*config.n_embd,config.n_embd)
                self.c_proj.NANOGPT_SCALE_INIT=1

        def forward(self,x):
                x=self.c_fc(x)
                x=self.gelu(x)
                x=self.c_proj(x)
                return x

class Block(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.ln_1=nn.LayerNorm(config.n_embd)
                self.attn=CausalAttention(config)
                self.ln_2=nn.LayerNorm(config.n_embd)
                self.mlp=MLP(config)
        def forward(self,x):
                x=x+self.attn(self.ln_1(x))
                x=x+self.mlp(self.ln_2(x))
                return x

@dataclass
class GPTConfig:
        vocab_size:int=50257
        block_size:int=1024
        n_head:int=12
        n_layer:int=12
        n_embd:int=768
class GPT(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.transformer=ModuleDict(dict(
                        wte=nn.Embedding(config.vocab_size,config.n_embd),
                        wpe=nn.Embedding(config.block_size,config.n_embd),
                        h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                        ln_f=nn.LayerNorm(config.n_embd),
                ))
                self.lm_head=nn.Linear(config.n_embd,config.vocab_size,bias=False)
                self.transformer.wte.weight=self.lm_head.weight
                self.apply(self.init_weights)
        def init_weights(self,module):
               std=0.02
               if isinstance(module,nn.Linear):
                      if hasattr(module, 'NANOGPT_SCALE_INIT'):
                                std*=(2*self.config.n_layer)**-0.5
                      torch.nn.init.normal_(module.weight,mean=0.0,std=std)
                      if module.bias is not None:
                          torch.nn.init.zeros_(module.bias)
               if isinstance(module,nn.Embedding):
                        torch.nn.init.normal_(module.weight,mean=0.0,std=std)
        def forward(self, idx, targets=None):
                B, T = idx.size()
                assert T <= self.config.block_size, f"Cannot forward sequence of length {T},block size is only {self.config.block_size}"
                pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
                pos_emb = self.transformer.wpe(pos)
                tok_emb = self.transformer.wte(idx)
                x = tok_emb + pos_emb
                for block in self.transformer.h:
                        x = block(x)
                x = self.transformer.ln_f(x)
                logits = self.lm_head(x)
                loss = None
                if targets is not None:
                        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1))
                return logits, loss
        def configure_optimizers(self,weight_decay,learning_rate):
                param_dict={pn:p for pn,p in self.named_parameters()}
                param_dict={pn:p for pn,p in param_dict.items() if p.requires_grad}
                decay_params = [p for n,p in param_dict.items() if p.dim() >= 2]
                no_decay_params = [p for n,p in param_dict.items() if p.dim() < 2]
                optim_groups = [
                        {"params": decay_params, "weight_decay": weight_decay},
                        {"params": no_decay_params, "weight_decay": 0.0},
                ]
                num_decay_params = sum(p.numel() for p in decay_params)
                num_no_decay_params = sum(p.numel() for p in no_decay_params)
                print(f"num decay params: {num_decay_params}, num no decay params: {num_no_decay_params}")
                fused_available='fused' in inspect.signature(torch.optim.AdamW).parameters
                use_fused = fused_available and torch.cuda.is_available()
                if use_fused:
                        print(f"using fused AdamW: {use_fused}")
                optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
                return optimizer
        @classmethod
        def from_pretrained(cls, model_type):
                """Loads pretrained GPT-2 model weights from huggingface"""
                assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
                print("loading weights from pretrained gpt: %s" % model_type)
                config_args = {
                        'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
                        'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
                        'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
                        'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
                }[model_type]
                config_args['vocab_size']=50257
                config_args['block_size']=1024
                config = GPTConfig(**config_args)
                model = GPT(config)
                sd = model.state_dict()
                sd_keys = sd.keys()
                sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')]
                model_hf = GPT2LMHeadModel.from_pretrained(model_type)
                sd_hf = model_hf.state_dict()
                sd_keys_hf = sd_hf.keys()
                sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
                sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]
                transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
                assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
                for k in sd_keys_hf:
                        if any(k.endswith(w) for w in transposed):
                                assert sd_hf[k].shape[::-1]==sd[k].shape
                                with torch.no_grad():
                                        sd[k].copy_(sd_hf[k].t())
                        else:
                                assert sd_hf[k].shape == sd[k].shape
                                with torch.no_grad():
                                        sd[k].copy_(sd_hf[k])
                return model

# def load_tokens(filename):
#     npt = np.load(filename)
#     npt = npt.astype(np.int32)
#     ptt = torch.tensor(npt, dtype=torch.long)
#     return ptt

class DataLoaderLite:
    def __init__(self, B, T, process_rank=0, num_processes=1, split="train"):
        self.B = B
        self.T = T
        self.current_position = 0
        if split=="train":
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized/train_tokens.bin"
        else:
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized/val_tokens.bin"
        with open(filepath, "rb") as f:
                tokens_np= np.frombuffer(f.read(),dtype=np.uint16)
        self.tokens = torch.tensor(tokens_np,dtype=torch.long)

        #IF WE WANT TO USE MULTIPLE PROCESSES, WE CAN USE THIS CODE
        # self.process_rank = process_rank
        # self.num_processes = num_processes
        # assert split in {'train', 'val'}
        # data_root = "/mnt/d/AI/GPT-2/fineweb_data/tokenized"
        # shards = os.listdir(data_root)
        # shards = [s for s in shards if split in s]
        # shards = sorted(shards)
        # shards = [os.path.join(data_root, s) for s in shards]
        # self.shards = shards
        # assert len(shards) > 0, f"no shards found for split {split}"
        # if self.process_rank == 0:
        #     print(f"found {len(shards)} shards for split {split}")
        # self.reset()

#     def reset(self):
#         self.current_shard = 0
#         self.tokens = load_tokens(self.shards[self.current_shard])
#         self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T
        if self.current_position + (B * T + 1) > len(self.tokens):
                self.current_position = 0
        buf = self.tokens[self.current_position : self.current_position+B*T+1]
        x = (buf[:-1]).view(B, T) # inputs
        y = (buf[1:]).view(B, T) # targets
        # advance the position in the tensor
        self.current_position += B * T # * self.num_processes
        # if loading the next batch would be out of bounds, advance to next shard
        # if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
        #     self.current_shard = (self.current_shard + 1) % len(self.shards)
        #     self.tokens = load_tokens(self.shards[self.current_shard])
        #     self.current_position = B * T * self.process_rank
        return x, y


/home/awais/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
#INFERENCE CODE
state_dict=torch.load('/mnt/d/AI/GPT-2/model_best_final.pt')
state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
model = GPT(GPTConfig(vocab_size=50304))
model.load_state_dict(state_dict)
model.to("cuda")
model.eval()

from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

prompt = "Philosophy is the study of fundamental questions"
tokens = tokenizer.encode(prompt)
tokens = torch.tensor([tokens], dtype=torch.long).to("cuda")

with torch.no_grad():
    for _ in range(100):
        logits, _ = model(tokens)
        probs = torch.softmax(logits[0, -1], dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

output = tokenizer.decode(tokens[0].tolist())
print(output)

Philosophy is the study of fundamental questions that illustrate evolutionary transformations from two energy states. The source of energy depends on the choices between postsynaptic and postsynaptic pathways. These pathways play key roles along the lines of the kinds of energy that influence movement, memory, and heat flux. swiftly, these pathways have been activated to produce free energy, some of the simplest adds up to their limiting integrations, the phenolic elements required for the process.
Microscopic Origins of the Electronicnetic Synthesis
 Hedgehog 2019-2016
